In [ ]:
BANDS_TO_USE = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']
save_dir_imgs = "./dataset/test/imgs"
save_dir_masks = "./dataset/test/masks/"

# **Global ML**

Illegal mines: gather locations and timestamps, download multispectral images with ee

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import rioxarray

import ee
import wxee
from dotenv import load_dotenv
load_dotenv()
ee.Authenticate()
ee.Initialize(project=os.getenv("EAI_PROJECT_ID"))

import numpy as np
from datetime import datetime, timedelta

In [ ]:
# Load the GeoPackage
gdf = gpd.read_file('./data/Global_ML/mining_area_data.gpkg')

gdf_metric = gdf.to_crs(epsg=3857) 
gdf_buffered_metric = gdf_metric.copy()
gdf_buffered_metric['geometry'] = gdf_metric.geometry.buffer(1000) # Add a 1km buffer for context
gdf_buffered = gdf_buffered_metric.to_crs(epsg=4326)

print(gdf_buffered.head())

/home/filippo/Scrivania/electiveAI/Spectral-Spatial-Anomaly-Detection/venv/lib/python3.10/site-packages/pyogrio/geopandas.py:382: UserWarning: More than one layer found in 'mining_area_data.gpkg': 'maus_polygons' (default), 'tang_polygons', 'maus_bboxes', 'tang_bboxes', 'preferred_polygons', 'preferred_bboxes', 'tiles', 'test_polygons_validated'. Specify layer parameter to avoid this warning.
  result = read_func(


   tile_id                                           geometry
0     2805  POLYGON ((141.74665 -12.51526, 141.74596 -12.5...
1     1612                                               None
2     2352  POLYGON ((116.81998 -8.95424, 116.82047 -8.953...
3     1167  MULTIPOLYGON (((17.90669 -31.25206, 17.91378 -...
4     1107  MULTIPOLYGON (((8.21934 34.37455, 8.21852 34.3...


In [ ]:
# download the mines multispectral images
for index, row in gdf_buffered.iterrows():
    minx, miny, maxx, maxy = row.geometry.bounds    # bounding box from the polygon geometry
    roi = ee.Geometry.Rectangle([minx, miny, maxx, maxy])
    
    # create a small window around the timestamp
    center_date = pd.to_datetime(row['timestamp'])
    start_date = (center_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    end_date = (center_date + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    try:
        # --------------- define the image collection
        s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi)
              .filterDate(start_date, end_date)
              # .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))    # discard images more than 30% cloudy
              .select(BANDS_TO_USE)
            )

        # Check if collection is empty before downloading
        if s2.size().getInfo() == 0:
            print(f"No image found for ROI {index} on {center_date}")
            continue

        # Sort by time and grab the exact first image to prevent pixel blending
        s2_image = s2.sort('system:time_start').first()

        # --------------- download the images (from the roi at the specified time interval) into the RAM
        # ts = wxee.TimeSeries(s2)    # create time series
        # fetch into RAM (Lazy Loading)
        # ds = ts.wx.to_xarray(region=roi, scale=10, progress=False)  # resampling at scale
        ds = s2_image.wx.to_xarray(region=roi, scale=10, crs="EPSG:3857", progress=False)


        # --------------- convert data to numpy
        # stack bands
        data = np.stack([ds[b].values for b in BANDS_TO_USE], axis=0)    # (Channels, H, W)
        img_array = np.nan_to_num(data) # clean up any missing edge pixels

        # --------------- rasterize the mask
        ds = ds.rio.write_crs("EPSG:3857") # Explicitly register CRS to prevent rioxarray crashes
        transform = ds.rio.transform()
        out_shape = (img_array.shape[1], img_array.shape[2]) # (H, W)
        # Burn the original unbuffered polygon (from 'gdf', not 'gdf_buffered') into the mask
        original_polygon = gdf_metric.iloc[index].geometry
        mask_array = rasterize(
            [(original_polygon, 1)], # Label the polygon area as 1
            out_shape=out_shape,
            transform=transform,
            fill=0, # Background is 0
            dtype='uint8'
        )

        np.save(os.path.join(save_dir_imgs, f"mines_{index}.npy"), img_array)
        np.save(os.path.join(save_dir_masks, f"mines_{index}_mask.npy"), mask_array)

        print(f"Saved ROI {index} - Image: {img_array.shape}, Mask: {mask_array.shape}")

    except Exception as e:
        print(f"Skipping ROI {index}, error: {e}")
        continue

# **hls burn scars**
Burn scars: gather locations and timestamps, download multispectral images with ee

In [ ]:
import os
import glob
import rasterio
from rasterio.warp import transform_bounds
import pandas as pd
from datetime import datetime, timedelta

import pandas as pd

import ee
import wxee
from dotenv import load_dotenv
load_dotenv()
ee.Authenticate()
ee.Initialize(project=os.getenv("EAI_PROJECT_ID"))

In [ ]:
def create_metadata_csv(dataset_dir, output_csv):
    """
    Extracts dates and geographic bounds from subsetted HLS .tif files.
    """
    data = []
    
    # Find all merged images (ignoring the masks for the metadata extraction)
    search_pattern = os.path.join(dataset_dir, '**', '*_merged.tif')
    tif_files = glob.glob(search_pattern, recursive=True)
    
    print(f"Found {len(tif_files)} files. Extracting metadata...")
    
    for filepath in tif_files:
        filename = os.path.basename(filepath)
        
        # Parse filename: subsetted_512x512_HLS.S30.T10SEH.2018190.v1.4_merged.tif
        parts = filename.split('.')
        tile_id = parts[2]       # e.g., T10SEH
        year_doy = parts[3]      # e.g., 2018190 (Year 2018, Day of Year 190)
        
        # Convert Year+DayOfYear to standard YYYY-MM-DD
        date_obj = datetime.strptime(year_doy, '%Y%j')
        date_str = date_obj.strftime('%Y-%m-%d')
        
        # Open the TIF to extract exact spatial bounds
        with rasterio.open(filepath) as src:
            bounds = src.bounds
            crs = src.crs
            
            # GEE needs standard EPSG:4326 (Lat/Lon)
            min_lon, min_lat, max_lon, max_lat = transform_bounds(crs, 'EPSG:4326', *bounds)
            
        data.append({
            'filename': filename,
            'mask_filename': filename.replace('_merged.tif', '.mask.tif'),
            'tile_id': tile_id,
            'date': date_str,
            'min_lon': min_lon,
            'min_lat': min_lat,
            'max_lon': max_lon,
            'max_lat': max_lat
        })
        
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"Saved metadata to {output_csv}")

# Run the extraction
create_metadata_csv('path/to/hls_burn_scars', 'burn_scars_metadata.csv')

In [ ]:
def download_gee_images_from_csv(csv_path, output_dir, bands_to_use):
    df = pd.read_csv(csv_path)
    
    for index, row in df.iterrows():
        # 1. Define Geometry
        roi = ee.Geometry.Rectangle([row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat']])
        
        # 2. Define Time Window (Exact date to Date + 1 day)
        t_start = row['date']
        t_end = (datetime.strptime(t_start, '%Y-%m-%d') + timedelta(days=1)).strftime('%Y-%m-%d')
        
        # 3. Query GEE for Sentinel-2 Level-2A
        s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi)
              .filterDate(t_start, t_end)
              .select(bands_to_use))
              
        # Check if an image actually exists for that exact day
        if s2.size().getInfo() == 0:
            print(f"Skipping {row['filename']}: No S2 image found in GEE for {t_start}.")
            continue
            
        try:
            # 4. Download and save locally
            print(f"Downloading data for {row['filename']} on {t_start}...")
            s2_mosaic = s2.mosaic().set('system:time_start', ee.Date(t_start).millis())
            s2_collection = ee.ImageCollection([s2_mosaic])

            ts = wxee.TimeSeries(s2_collection)
            
            # Download as an xarray dataset (scale=10 for 10m resolution)
            ds = ts.wx.to_xarray(region=roi, scale=10, progress=False)
            
            # Save it locally as a NetCDF or convert to Numpy/TIF
            out_filename = "burnscars_" + row['filename'].replace('.tif', '_13bands.nc')
            out_file = os.path.join(output_dir, out_filename)
            ds.to_netcdf(out_file)
            
        except Exception as e:
            print(f"Error downloading {row['filename']}: {e}")

download_gee_images_from_csv('burn_scars_metadata.csv', save_dir, BANDS_TO_USE)